# DS2002 · SQL Challenge Set

**Lab — 2026-09-11 · Fall 2026**  

---

## Lab 03 — SQL Challenge Set

Seven questions, one query each. Every query has to produce the right answer when the notebook is run from a fresh kernel, top to bottom.

Two rules that matter as much as getting the answer:

- **Check the row count** against what you expect before you believe a result.
- **Decide what to do about the untagged track and the unplayed tracks.** Several of these questions have a defensible answer either way; what is not defensible is not noticing they exist.

In [1]:
import sqlite3, pandas as pd
conn = sqlite3.connect(':memory:')
cur = conn.cursor()
cur.executescript('''
CREATE TABLE artists (artist_id INTEGER PRIMARY KEY, name TEXT, country TEXT);
CREATE TABLE tracks (track_id INTEGER PRIMARY KEY, title TEXT, artist_id INTEGER, genre TEXT, seconds INTEGER);
CREATE TABLE plays (play_id INTEGER PRIMARY KEY, track_id INTEGER, user TEXT, played_on TEXT);
INSERT INTO artists VALUES
 (1,'Nova Waves','US'),(2,'The Blue Ridge','US'),(3,'Kestrel','UK'),(4,'Marisol','ES');
INSERT INTO tracks VALUES
 (10,'Skyline',1,'Pop',201),(11,'Undertow',1,'Pop',240),(12,'Foothills',2,'Folk',185),
 (13,'Aurora',3,'Electronic',300),(14,'Nightfall',3,'Electronic',275),(15,'Sol',4,'Latin',210),
 (16,'Coastline',2,'Folk',199),(17,'Ridgeline',2,'Folk',225),(18,'Untitled Demo',3,NULL,150);
INSERT INTO plays VALUES
 (100,10,'ava','2026-09-01'),(101,10,'ben','2026-09-01'),(102,13,'ava','2026-09-02'),
 (103,13,'cara','2026-09-02'),(104,14,'ben','2026-09-03'),(105,12,'ava','2026-09-03'),
 (106,15,'dan','2026-09-04'),(107,10,'cara','2026-09-04'),(108,13,'dan','2026-09-05'),
 (109,16,'ava','2026-09-05'),(110,11,'ben','2026-09-06');
''')
conn.commit()

def q(sql):
    return pd.read_sql_query(sql, conn)
print('ready')

ready


### Q1 — Every track with its artist's name and country.

*Expected: 9 rows, one per track.*

In [2]:
q1= q('''
SELECT t.track_id, t.title, a.name, a.country
FROM tracks t
JOIN artists a
ON t.artist_id = a.artist_id
''')

q1

,track_id,title,name,country
0,10,Skyline,Nova Waves,US
1,11,Undertow,Nova Waves,US
2,12,Foothills,The Blue Ridge,US
3,13,Aurora,Kestrel,UK
4,14,Nightfall,Kestrel,UK
5,15,Sol,Marisol,ES
6,16,Coastline,The Blue Ridge,US
7,17,Ridgeline,The Blue Ridge,US
8,18,Untitled Demo,Kestrel,UK


Here we're selecting `track_id` and `title` from the `tracks` table, as well as `name` and `country` from the `artists` table to get all required information. Because these columns come from separate tables, we must do an inner join that matches on `artist_id`.

### Q2 — Which genre has the longest average track length?

Return the genre and the average, not just the name.

In [3]:
q2= q('''
SELECT genre, AVG(seconds) as avg_seconds
FROM tracks
WHERE genre IS NOT NULL
GROUP BY genre
ORDER BY avg_seconds DESC
LIMIT 1
''')

q2

,genre,avg_seconds
0,Electronic,287.5


This groups `tracks` by `genre` and computes the average seconds for each genre, excluding the untagged track (`genre IS NULL`) since a missing genre isn't a real genre to rank. It then sorts the results descending by that average and takes the top row to find the genre with the longest average track length.

### Q3 — For each user: how many plays, and how many distinct tracks?

Someone who played one track four times is a different listener from someone who played four different tracks. Your result should make that visible.

In [4]:
q3= q('''
SELECT user, COUNT(*) AS plays, COUNT(DISTINCT track_id) AS distinct_tracks
FROM plays
GROUP BY user
''')

q3


,user,plays,distinct_tracks
0,ava,4,4
1,ben,3,3
2,cara,2,2
3,dan,2,2


This groups the `plays` table by `user`, then counts the total rows per group with `COUNT(*)` to get the total number of plays, and separately counts unique `track_id` values with `COUNT(DISTINCT track_id)` to get `distinct_tracks`.

### Q4 — Which tracks have never been played?

*Expected: 2 rows.* Hint: `LEFT JOIN` and then keep the rows where the right side came back `NULL`.

In [5]:
q4= q('''
SELECT t.track_id, t.title
FROM tracks t
LEFT JOIN plays p
ON t.track_id = p.track_id
WHERE p.track_id IS NULL
''')

q4

,track_id,title
0,17,Ridgeline
1,18,Untitled Demo


This code uses a left-join, joining `tracks` to `plays` so every track is kept even if it has no matching plays. Then I filtered for rows where `p.track_id IS NULL`, finding rows where the join found no match, so that track was never played.

### Q5 — Rank artists by total listening time.

Sum the seconds actually listened across all plays, most to least, and include a minutes column rounded to one decimal.

In [6]:
q5= q('''
SELECT a.name AS artist, SUM(t.seconds) AS total_seconds, ROUND(SUM(t.seconds) / 60.0, 1) AS total_minutes
FROM plays p
JOIN tracks t ON p.track_id = t.track_id
JOIN artists a ON t.artist_id = a.artist_id
GROUP BY a.artist_id
ORDER BY total_seconds DESC;
''')
q5

,artist,total_seconds,total_minutes
0,Kestrel,1175,19.6
1,Nova Waves,843,14.1
2,The Blue Ridge,384,6.4
3,Marisol,210,3.5


This joins `artists` to `tracks` and then to `plays`, so each play contributes its track's seconds once. This results in a track that is played multiple times being counted multiple times toward its artist's total. Additionally, a track that was never played contributes nothing, since it has no row in `plays` to join from. Summing seconds per artist and dividing by 60 (rounded to one decimal) turns that into total minutes, then the results are sorted descending to rank artists from most to least listened.

### Q6 — Which tracks are missing a genre?

Return the track id and title. Then, in a comment, say what `WHERE genre != 'Pop'` would have done to these rows and why.

In [7]:
q6= q('''
SELECT track_id, title
FROM tracks
WHERE genre IS NULL;
''')

q6

,track_id,title
0,18,Untitled Demo


This filters tracks for rows where `genre IS NULL`, since a regular comparison operator can't detect missing values. If we used `WHERE genre != 'Pop'`, this would exclude this row instead of including it, because `NULL != 'Pop'` results with an unknown rather than true, and `WHERE` only keeps rows that evaluate to true.

### Q7 — Plays per day.

`played_on` is stored as text like `'2026-09-01'`. Count plays per date, earliest first, and include the number of distinct users active that day.

In [8]:
q7= q('''
SELECT played_on, COUNT(*) AS total_plays, COUNT(DISTINCT user) AS distinct_users
FROM plays
GROUP BY played_on
ORDER BY played_on
''')

q7

,played_on,total_plays,distinct_users
0,2026-09-01,2,2
1,2026-09-02,2,2
2,2026-09-03,2,2
3,2026-09-04,2,2
4,2026-09-05,2,2
5,2026-09-06,1,1


This groups the `plays` table by `played_on`, then counts total rows/plays per date with `COUNT(*)` as well as unique users with `COUNT(DISTINCT user)`. Dates are stored as 'YYYY-MM-DD' text, so sorting them alphabetically will sort them chronologically.

### Validate your work

**TODO:** uncomment these and make them pass. Assign your query results to the variables as you go — for example `q1 = q('''...''')`.

In [9]:
assert len(q1) == 9, 'Q1 should return one row per track'
assert len(q4) == 2, 'Q4: two tracks have never been played'
assert q3['plays'].sum() == 11, 'Q3 should account for all 11 plays'
print('checks passed.')

checks passed.


### Write-up

Pick the query that gave you the most trouble and explain what you had wrong before you had it right. Name the specific misunderstanding — "I put the aggregate in WHERE" or "I used an inner join and lost the tracks with no plays" — not "it was confusing."

When I initially attempted Q2, my code looked like this:
```
q('''
SELECT genre, AVG(seconds)
FROM tracks
WHERE AVG(seconds) = MAX(AVG(seconds))
GROUP BY genre
''')
```
This gave me an error because I was trying to filter to get the genre with the highest average track length using `WHERE`, but `WHERE` is trying to run before grouping and aggregation happen. So at that point, `AVG(seconds)` doesn't exist yet, since there's no group to average over. Aggregates only make sense after `GROUP BY` had formed the groups, so instead I used `ORDER BY avg_seconds DESC LIMIT 1` to get the top row instead of trying to filter for the max.
